# Checagens: o modelo está bom demais?

Os resultados das Etapas A e B são altos (MCC ~0,97). Antes de confiar neles,
este notebook testa explicações alternativas para o acerto alto. Nada aqui
altera os dados: tudo roda em memória, só com treino e validação (o teste
continua intocado até a Etapa E).

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

dados = np.load("../dados_processados/etapa2.npz", allow_pickle=True)
X_todos = dados["X_simples"].astype(np.float32)  # já normalizado com média/desvio do treino (Etapa 2)
y_especie = dados["y_simples"]
pos_treino = dados["pos_treino_simples"]
pos_val = dados["pos_val_simples"]

## Funções reaproveitadas das Etapas A e B

Mesma MLP do A3 (5-8-1, `Tanh`, saída sigmoide, `BCELoss`, Adam lr 0,01,
mini-batch 64, 400 épocas, pesos do mínimo da perda de validação) e as mesmas
métricas à mão do Slides_6.

In [2]:
def cria_mlp(n_entradas, n_ocultos=8):
    return nn.Sequential(
        nn.Linear(n_entradas, n_ocultos), nn.Tanh(),
        nn.Linear(n_ocultos, 1), nn.Sigmoid(),
    )


def treina(modelo, t_Xtr, t_ytr, t_Xva, t_yva, n_epocas=400, semente=42):
    torch.manual_seed(semente)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    criterio = nn.BCELoss()
    loader = DataLoader(TensorDataset(t_Xtr, t_ytr), batch_size=64, shuffle=True)
    melhor, melhores_pesos = float("inf"), None
    for _ in range(n_epocas):
        modelo.train()
        for xb, yb in loader:
            otimizador.zero_grad()
            perda = criterio(modelo(xb), yb)
            perda.backward()
            otimizador.step()
        modelo.eval()
        with torch.no_grad():
            pv = criterio(modelo(t_Xva), t_yva).item()
        if pv < melhor:
            melhor, melhores_pesos = pv, copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(melhores_pesos)
    return modelo


def matriz_confusao(y_real, y_prob, limiar=0.5):
    y_pred = (y_prob >= limiar).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_real == 1)))
    fp = int(np.sum((y_pred == 1) & (y_real == 0)))
    tn = int(np.sum((y_pred == 0) & (y_real == 0)))
    fn = int(np.sum((y_pred == 0) & (y_real == 1)))
    return tp, fp, tn, fn


def mcc(tp, fp, tn, fn):
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return ((tp * tn) - (fp * fn)) / den if den > 0 else 0.0


def probabilidades(modelo, X):
    modelo.eval()
    with torch.no_grad():
        return modelo(torch.from_numpy(X)).numpy().flatten()

## Checagem 1: rótulos embaralhados

Embaralha os rótulos do treino (e os da validação usados só para escolher a
época), quebrando a relação entre medidas e espécie. A avaliação final usa os
rótulos **verdadeiros** da validação. Se o pipeline estiver limpo, não há o que
aprender e o MCC deve ficar perto de 0. Se ficar alto, a espécie está vazando
por outro caminho (ordem das linhas, normalização, escolha do checkpoint).

Comparação: o mesmo treino com os rótulos certos. Dois problemas: *lavali* ×
resto (Etapa A) e *lavali* × *temminckii* (Etapa B), 3 embaralhamentos cada.

In [3]:
def monta_problema(nome):
    if nome == "lavali × resto":
        tr, va = pos_treino, pos_val
        pos = "Myotis_lavali"
    else:  # lavali × temminckii
        par = ["Myotis_lavali", "Molossops_temminckii"]
        tr = pos_treino[np.isin(y_especie[pos_treino], par)]
        va = pos_val[np.isin(y_especie[pos_val], par)]
        pos = "Molossops_temminckii"
    y = lambda p: (y_especie[p] == pos).astype(np.float32)
    return X_todos[tr], y(tr), X_todos[va], y(va)


T = lambda a: torch.from_numpy(a)
linhas = []
for problema in ["lavali × resto", "lavali × temminckii"]:
    Xtr, ytr, Xva, yva = monta_problema(problema)

    torch.manual_seed(42)
    m = treina(cria_mlp(5), T(Xtr), T(ytr).unsqueeze(1), T(Xva), T(yva).unsqueeze(1))
    linhas.append({"problema": problema, "rótulos": "verdadeiros", "repetição": 0,
                   "MCC na validação": mcc(*matriz_confusao(yva, probabilidades(m, Xva)))})

    for rep in range(3):
        rng = np.random.default_rng(100 + rep)
        ytr_emb = rng.permutation(ytr)  # cópia embaralhada; ytr original não muda
        yva_emb = rng.permutation(yva)  # só para escolher a época
        torch.manual_seed(rep)
        m = treina(cria_mlp(5), T(Xtr), T(ytr_emb).unsqueeze(1), T(Xva), T(yva_emb).unsqueeze(1), semente=rep)
        prob = probabilidades(m, Xva)
        linhas.append({"problema": problema, "rótulos": "embaralhados", "repetição": rep + 1,
                       "MCC na validação": mcc(*matriz_confusao(yva, prob)),
                       "fração prevista positiva": float((prob >= 0.5).mean())})

df_emb = pd.DataFrame(linhas).round(3)
df_emb

,problema,rótulos,repetição,MCC na validação,fração prevista positiva
0,lavali × resto,verdadeiros,0,0.972,NaN
1,lavali × resto,embaralhados,1,0.096,0.977
2,lavali × resto,embaralhados,2,-0.020,1.000
3,lavali × resto,embaralhados,3,0.024,1.000
4,lavali × temminckii,verdadeiros,0,0.963,NaN
5,lavali × temminckii,embaralhados,1,0.000,0.000
6,lavali × temminckii,embaralhados,2,0.000,0.000
7,lavali × temminckii,embaralhados,3,0.000,0.000
